# Vilier Colab Runner

Notebook nay mount Google Drive, clone/pull repo, cai dependencies, chay `bash run.sh` voi input lay tu Drive va output ghi lai Drive.

In [1]:
# Sua cac gia tri nay truoc khi chay neu can.
REPO_URL = "https://github.com/ngocbao220/vilier.git"
BRANCH = "main"
PROJECT_DIR = "/content/vilier"

# Dat file audio trong Google Drive, vi du: MyDrive/vilier/input/real.wav
DRIVE_AUDIO_PATH = "/content/drive/MyDrive/VDT-TurnTaking/inputs/haveasip_khanhvi_5m_1.mp3"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/VDT-TurnTaking/outputs"

# Bat ASR neu muon chay PhoWhisper tren GPU Colab. Mac dinh giu false de tach speaker truoc.
ENABLE_ASR = False
ENABLE_STATE_LABELING = False
DRY_RUN = False
DIARIZATION_BACKEND = "sortformer"  # Hỗ trợ: "sortformer" hoặc "pixit"


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os
import subprocess
from pathlib import Path

project_dir = Path(PROJECT_DIR)
if project_dir.exists():
    subprocess.run(["git", "-C", str(project_dir), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(project_dir), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(project_dir)], check=True)

os.chdir(project_dir)
print("Repo:", project_dir)


Repo: /content/vilier


In [4]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-r', 'requirements.txt'], returncode=0)

In [5]:
DIARIZATION_BACKEND = "pyannote/speaker-diarization-community-1"  # Hỗ trợ: "sortformer", "pixit", "diarizen"

In [ ]:
import json
import os
import sys
from pathlib import Path

try:
    from google.colab import userdata
except Exception:
    userdata = None

def set_secret_from_colab(secret_name, env_name):
    if os.environ.get(env_name) or userdata is None:
        return
    try:
        value = userdata.get(secret_name)
    except Exception:
        value = None
    if value:
        os.environ[env_name] = value

set_secret_from_colab("HUGGINGFACE_TOKEN", "HUGGINGFACE_TOKEN")
set_secret_from_colab("HF_TOKEN", "HF_TOKEN")
set_secret_from_colab("DASHSCOPE_API_KEY", "DASHSCOPE_API_KEY")

audio_path = Path(DRIVE_AUDIO_PATH)
output_dir = Path(DRIVE_OUTPUT_DIR)
if not audio_path.exists():
    raise FileNotFoundError(f"Drive audio not found: {audio_path}")
output_dir.mkdir(parents=True, exist_ok=True)

with open("config.json", "r", encoding="utf-8") as handle:
    config = json.load(handle)

config.setdefault("entrypoint", {})["input_path"] = str(audio_path)
config.setdefault("entrypoint", {})["output_path"] = str(output_dir)
config.setdefault("runtime", {})["dry_run"] = bool(DRY_RUN)
config.setdefault("diarization", {})["device"] = "cuda"
config["diarization"]["backend"] = DIARIZATION_BACKEND
if DIARIZATION_BACKEND == "sortformer":
        config["diarization"]["model"] = "nvidia/diar_sortformer_4spk-v1"
elif DIARIZATION_BACKEND == "pixit":
        config["diarization"]["model"] = "pyannote/speech-separation-ami-1.0"
elif DIARIZATION_BACKEND == "pyannote/speaker-diarization-community-1":
        config["diarization"]["model"] = "pyannote/speaker-diarization-community-1"
config.setdefault("asr", {})["enabled"] = bool(ENABLE_ASR)
config.setdefault("asr", {})["device"] = 0 if ENABLE_ASR else "cpu"
config.setdefault("state_labeling", {})["enabled"] = bool(ENABLE_STATE_LABELING)

config_path = Path("config.colab.json")
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

os.environ["CONFIG_PATH"] = str(config_path)
os.environ["INPUT_PATH"] = str(audio_path)
os.environ["OUTPUT_PATH"] = str(output_dir)
os.environ["PYTHON_BIN"] = sys.executable
if DRY_RUN:
    os.environ["DRY_RUN"] = "1"

print("CONFIG_PATH=", os.environ["CONFIG_PATH"])
print("INPUT_PATH=", os.environ["INPUT_PATH"])
print("OUTPUT_PATH=", os.environ["OUTPUT_PATH"])
print("ENABLE_ASR=", ENABLE_ASR)
print("ENABLE_STATE_LABELING=", ENABLE_STATE_LABELING)


In [ ]:
!hf auth login --token YOUR_HF_TOKEN

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Error: Invalid user token.
Hint: set HF_DEBUG=1 as environment variable for full traceback.


In [ ]:
!pip install -U "git+https://github.com/BUTSpeechFIT/DiariZen.git#egg=pyannote-audio&subdirectory=pyannote-audio"

  Cloning https://github.com/BUTSpeechFIT/DiariZen.git to /tmp/pip-install-rf695oha/pyannote-audio_3fef706a96eb4a42ad5804926aaf63ec
  Running command git clone --filter=blob:none --quiet https://github.com/BUTSpeechFIT/DiariZen.git /tmp/pip-install-rf695oha/pyannote-audio_3fef706a96eb4a42ad5804926aaf63ec
  Resolved https://github.com/BUTSpeechFIT/DiariZen.git to commit 844f5555b0a98acd0931511fc641a8c5b8ba92c7
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyannote-audio: filename=pyannote_audio-3.1.1-py2.py3-none-any.whl size=868660 sha256=c6b9ceb56bfd19e57c2018879aef19675e91e895f868428abf5e3b13bad3ee0d
  Stored in directory: /tmp/pip-ephem-whee

In [ ]:
!bash run.sh

[INFO] Pipeline component usage
| Component          | Enabled | Backend          | Model                               |
| ------------------ | ------- | ---------------- | ----------------------------------- |
| VAD                | V       | silero           | silero_vad                          |
| Diarization        | V       | diarizen         | BUT-FIT/diarizen-wavlm-large-s80-md |
| Music separation   | X       | demucs           | htdemucs                            |
| Overlap separation | V       | sepreformer      | SepReformer_Large_DM_WHAMR          |
| ASR                | X       | phowhisper_local | vinai/PhoWhisper-large              |
| State labeling     | X       | qwen             | qwen3.8-max                         |
[INFO] Vilier pipeline
└── batch
    ├── input_path=../drive/MyDrive/VDT-TurnTaking/inputs/haveasip_khanhvi_5m_1.mp3
    ├── output_path=../drive/MyDrive/VDT-TurnTaking/outputs
    ├── log_dir=logs/2026-08-26
    ├── state_dir=../drive/MyDrive/VDT-

In [ ]:
from pathlib import Path

audio_id = Path(DRIVE_AUDIO_PATH).stem
result_dir = Path(DRIVE_OUTPUT_DIR) / audio_id
print("Result dir:", result_dir)
for path in sorted(result_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(result_dir))


Result dir: /content/drive/MyDrive/VDT-TurnTaking/outputs/haveasip_khanhvi_5m_1
asr_audio/SPEAKER_00/audio_00001.wav
asr_audio/SPEAKER_00/audio_00002.wav
asr_audio/SPEAKER_00/audio_00003.wav
asr_audio/SPEAKER_00/audio_00004.wav
asr_audio/SPEAKER_00/audio_00005.wav
asr_audio/SPEAKER_00/audio_00006.wav
asr_audio/SPEAKER_00/audio_00007.wav
asr_audio/SPEAKER_00/audio_00008.wav
asr_audio/SPEAKER_00/audio_00009.wav
asr_audio/SPEAKER_00/audio_00010.wav
asr_audio/SPEAKER_00/audio_00011.wav
asr_audio/SPEAKER_00/audio_00012.wav
asr_audio/SPEAKER_00/audio_00013.wav
asr_audio/SPEAKER_00/audio_00014.wav
asr_audio/SPEAKER_00/audio_00015.wav
asr_audio/SPEAKER_00/audio_00016.wav
asr_audio/SPEAKER_00/audio_00017.wav
asr_audio/SPEAKER_00/audio_00018.wav
asr_audio/SPEAKER_01/audio_00001.wav
asr_audio/SPEAKER_01/audio_00002.wav
asr_audio/SPEAKER_01/audio_00003.wav
asr_audio/SPEAKER_01/audio_00004.wav
asr_audio/SPEAKER_01/audio_00005.wav
asr_audio/SPEAKER_01/audio_00006.wav
asr_audio/SPEAKER_01/audio_00007